# YTMusicAPI: one artist at a time

Open in VS Code with **Polyglot Notebooks**, choose the **.NET (C#)** kernel, and run all cells once. Then change `artistName` in the search cell and rerun that cell and the cells below it.

Uses the app's pinned YTMusicAPI **1.0.10**. No API key, Tavily request, or database connection is needed. Searches make live requests to YouTube Music.

The tables preserve returned order and show what `YouTubeMusicEnricher` would select. A matching name does **not** verify artist identity: inspect the channel and song links, especially when multiple channels share a name. Results cover the **first search page**, matching the app; the final cell includes full package response objects and continuation metadata (not the underlying HTTP JSON).


In [1]:
#r "nuget: YTMusicAPI, 1.0.10"

using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Text;
using System.Text.Json;
using System.Text.RegularExpressions;
using System.Threading;
using System.Threading.Tasks;
using YTMusicAPI;
using YTMusicAPI.Model;

var search = new SearchClient();

// Copied from CorrectionWorker.NormalizeArtist; keep in sync when app rules change.
string NormalizeArtist(string value)
{
    var ascii = new string(value.Normalize(NormalizationForm.FormKD)
        .Where(c => c <= 127).ToArray()).ToLowerInvariant().Trim();
    return Regex.Replace(Regex.Replace(ascii, @"^the\s+", ""), "[^a-z0-9]+", "");
}

async Task<T> RunSearch<T>(string label, Func<CancellationToken, Task<T>> action) where T : class
{
    using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(45));
    var elapsed = Stopwatch.StartNew();
    try
    {
        var result = await action(timeout.Token);
        Console.WriteLine($"{label}: {elapsed.Elapsed.TotalSeconds:F2}s");
        return result;
    }
    catch (Exception ex)
    {
        Console.WriteLine($"{label} FAILED after {elapsed.Elapsed.TotalSeconds:F2}s:\n{ex}");
        return null;
    }
}


Installed Packages YTMusicAPI, 1.0.10

## Search — edit this artist name

Both searches run even if no artist matches so you can inspect the package's track output. Each search has its own 45-second timeout. The app instead shares 45 seconds across both calls and skips track search when no channel matches.


In [6]:
var artistName = "Can't Stop Talking"; // Change this and rerun this cell and everything below it.

artistName = artistName.Trim();
if (string.IsNullOrWhiteSpace(artistName) || NormalizeArtist(artistName).Length == 0)
    throw new ArgumentException("Enter an artist name that the app can normalize.");

Console.WriteLine($"Artist: {artistName} | normalized: {NormalizeArtist(artistName)}");
Console.WriteLine($"Search started (UTC): {DateTimeOffset.UtcNow:O}");

// Reset both responses before making requests so a failed rerun cannot show an old artist's data.
SearchingResult<YTMusicAPI.Model.Domain.Artist> artistResponse = null;
SearchingResult<YTMusicAPI.Model.Domain.Track> trackResponse = null;
artistResponse = await RunSearch("Artist/channel search", token =>
    search.SearchArtistsChannelsAsync(new QueryRequest { Query = artistName }, token));
trackResponse = await RunSearch("Track search", token =>
    search.SearchTracksAsync(new QueryRequest { Query = artistName }, token));


Artist: Can't Stop Talking | normalized: cantstoptalking
Search started (UTC): 2026-10-23T17:10:04.0794869+00:00
Artist/channel search: 0.43s
Track search: 0.62s


## Channel candidates and the app's selection

`SelectedByApp` is the **first** normalized-name match with a valid `UC...` ID. Every candidate stays visible, including rejected candidates. An error in either search is printed above; an empty table after an error is not a successful search with no results.


In [7]:
var artistResults = artistResponse?.Result?.ToArray()
    ?? Array.Empty<YTMusicAPI.Model.Domain.Artist>();
var trackResults = trackResponse?.Result?.ToArray()
    ?? Array.Empty<YTMusicAPI.Model.Domain.Track>();
var normalizedName = NormalizeArtist(artistName);
var matchingChannels = artistResults
    .Where(a => NormalizeArtist(a.Title ?? "") == normalizedName)
    .Where(a => Regex.IsMatch(a.Id ?? "", @"\AUC[A-Za-z0-9_-]+\z"))
    .ToArray();
var selectedChannel = matchingChannels.Select(a => a.Id).FirstOrDefault();
var selectedRank = Array.FindIndex(artistResults, a =>
    NormalizeArtist(a.Title ?? "") == normalizedName &&
    Regex.IsMatch(a.Id ?? "", @"\AUC[A-Za-z0-9_-]+\z"));

Console.WriteLine($"Returned: {artistResults.Length} artists/channels, {trackResults.Length} tracks.");
Console.WriteLine($"Normalized-name candidates: {matchingChannels.Length}; distinct matching channel IDs: {matchingChannels.Select(a => a.Id).Distinct().Count()}.");
if (matchingChannels.Select(a => a.Id).Distinct().Count() > 1)
    Console.WriteLine("AMBIGUOUS: the app takes the first channel; verify which artist it belongs to.");
Console.WriteLine(selectedChannel == null
    ? "App outcome: no matching artist channel; enrichment would fail."
    : $"Selected channel: https://www.youtube.com/channel/{selectedChannel}");

display(artistResults.Select((a, i) => new {
    Rank = i + 1,
    a.Title,
    NormalizedTitle = NormalizeArtist(a.Title ?? ""),
    a.Id,
    NameMatches = NormalizeArtist(a.Title ?? "") == normalizedName,
    ValidChannelId = Regex.IsMatch(a.Id ?? "", @"\AUC[A-Za-z0-9_-]+\z"),
    SelectedByApp = i == selectedRank,
    ChannelUrl = a.Url
}).ToArray());


Returned: 2 artists/channels, 20 tracks.
Normalized-name candidates: 2; distinct matching channel IDs: 2.
AMBIGUOUS: the app takes the first channel; verify which artist it belongs to.
Selected channel: https://www.youtube.com/channel/UCS2REA-szsJVdsSZJp2roHA


index value 0 { Rank = 1, Title = CAN'T STOP TALKING, NormalizedTitle = cantstoptalking, Id = UCS2REA-szsJVdsSZJp2roHA, NameMatches = True, ValidChannelId = True, SelectedByApp = True, ChannelUrl = https://music.youtube.com/channel/UCS2REA-szsJVdsSZJp2roHA } Rank 1 Title CAN'T STOP TALKING NormalizedTitle cantstoptalking Id UCS2REA-szsJVdsSZJp2roHA NameMatches True ValidChannelId True SelectedByApp True ChannelUrl https://music.youtube.com/channel/UCS2REA-szsJVdsSZJp2roHA 1 { Rank = 2, Title = CAN'T STOP TALKING, NormalizedTitle = cantstoptalking, Id = UCO4ec785eyIi_tD2GzrsyWQ, NameMatches = True, ValidChannelId = True, SelectedByApp = False, ChannelUrl = https://music.youtube.com/channel/UCO4ec785eyIi_tD2GzrsyWQ } Rank 2 Title CAN'T STOP TALKING NormalizedTitle cantstoptalking Id UCO4ec785eyIi_tD2GzrsyWQ NameMatches True ValidChannelId True SelectedByApp False ChannelUrl https://music.youtube.com/channel/UCO4ec785eyIi_tD2GzrsyWQ

## Track decisions and proposed links

The app keeps tracks whose `AuthorChannelId` equals the selected channel, rejects invalid IDs, removes duplicate video IDs, and takes the first three in returned order. Author text alone is not used for matching. These are search results, not a ranked list of the artist's top songs.


In [8]:
var seenTracks = new HashSet<string>();
var keptTrackIds = new List<string>();
var trackDecisions = trackResults.Select((t, i) => {
    var reason = selectedChannel == null ? "No selected channel"
        : t.AuthorChannelId != selectedChannel ? "Different author channel"
        : !Regex.IsMatch(t.Id ?? "", @"\A[A-Za-z0-9_-]+\z") ? "Invalid video ID"
        : !seenTracks.Add(t.Id) ? "Duplicate video ID"
        : keptTrackIds.Count >= 3 ? "Beyond first 3 matching tracks"
        : "KEEP";
    if (reason == "KEEP") keptTrackIds.Add(t.Id);
    return new {
        Rank = i + 1, t.Title, t.Author, t.AuthorChannelId,
        VideoId = t.Id, Decision = reason, TrackUrl = t.Url
    };
}).ToArray();

display(trackDecisions);

if (artistResponse == null || trackResponse == null)
    Console.WriteLine("A search failed; inspect the exception above before interpreting these results.");
else if (selectedChannel != null && keptTrackIds.Count == 0)
    Console.WriteLine("App outcome: no tracks matched the artist channel; existing links would be preserved.");
else if (selectedChannel != null)
{
    Console.WriteLine($"App would propose {keptTrackIds.Count} track link(s):");
    display(trackResults.Where(t => keptTrackIds.Contains(t.Id)).DistinctBy(t => t.Id)
        .Select(t => new {
            Platform = "youtube_music",
            Url = $"https://music.youtube.com/watch?v={t.Id}",
            Label = $"{artistName} - {t.Title}"
        }).ToArray());
}


index value 0 { Rank = 1, Title = Can't Stop Talking, Author = Hillsong Worship, AuthorChannelId = UClLLt3Gfr07OVs3qgWrOWjQ, VideoId = YALnDUy_x8o, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=YALnDUy_x8o } Rank 1 Title Can't Stop Talking Author Hillsong Worship AuthorChannelId UClLLt3Gfr07OVs3qgWrOWjQ VideoId YALnDUy_x8o Decision Different author channel TrackUrl https://music.youtube.com/watch?v=YALnDUy_x8o 1 { Rank = 2, Title = Cool People Are Arseholes, Author = CAN'T STOP TALKING, AuthorChannelId = UCO4ec785eyIi_tD2GzrsyWQ, VideoId = 0UkcxD1hH6k, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=0UkcxD1hH6k } Rank 2 Title Cool People Are Arseholes Author CAN'T STOP TALKING AuthorChannelId UCO4ec785eyIi_tD2GzrsyWQ VideoId 0UkcxD1hH6k Decision Different author channel TrackUrl https://music.youtube.com/watch?v=0UkcxD1hH6k 2 { Rank = 3, Title = Hot Stepper, Author = CAN'T STOP TALKING, AuthorChannelId = UCO4ec785eyIi_tD2GzrsyWQ, VideoId = YLPIrn2Ph_o, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=YLPIrn2Ph_o } Rank 3 Title Hot Stepper Author CAN'T STOP TALKING AuthorChannelId UCO4ec785eyIi_tD2GzrsyWQ VideoId YLPIrn2Ph_o Decision Different author channel TrackUrl https://music.youtube.com/watch?v=YLPIrn2Ph_o 3 { Rank = 4, Title = I Can't Stop Talking, Author = Nomeansno, AuthorChannelId = UCB6LN9qDURr9WmEAL-s092Q, VideoId = JpAlcljOKjc, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=JpAlcljOKjc } Rank 4 Title I Can't Stop Talking Author Nomeansno AuthorChannelId UCB6LN9qDURr9WmEAL-s092Q VideoId JpAlcljOKjc Decision Different author channel TrackUrl https://music.youtube.com/watch?v=JpAlcljOKjc 4 { Rank = 5, Title = Can't Stop Talking, Author = The Struts, AuthorChannelId = UC6EQ-tdoGO0kSxvoPgyAsKw, VideoId = m6OTCGkQ9Nc, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=m6OTCGkQ9Nc } Rank 5 Title Can't Stop Talking Author The Struts AuthorChannelId UC6EQ-tdoGO0kSxvoPgyAsKw VideoId m6OTCGkQ9Nc Decision Different author channel TrackUrl https://music.youtube.com/watch?v=m6OTCGkQ9Nc 5 { Rank = 6, Title = Can't Stop Talking (Live) (feat. Integrity's Hosanna! Music & Darlene Zschech), Author = Hillsong Worship, AuthorChannelId = UClLLt3Gfr07OVs3qgWrOWjQ, VideoId = W8c2TJ-mkfo, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=W8c2TJ-mkfo } Rank 6 Title Can't Stop Talking (Live) (feat. Integrity's Hosanna! Music & Darlene Zschech) Author Hillsong Worship AuthorChannelId UClLLt3Gfr07OVs3qgWrOWjQ VideoId W8c2TJ-mkfo Decision Different author channel TrackUrl https://music.youtube.com/watch?v=W8c2TJ-mkfo 6 { Rank = 7, Title = Can't Stop Singing (From "Teen Beach Movie"/Soundtrack Version), Author = Brady, AuthorChannelId = UCy95D_qGLG-eANxuO9DQEzA, VideoId = 6cWZWq5QQCQ, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=6cWZWq5QQCQ } Rank 7 Title Can't Stop Singing (From "Teen Beach Movie"/Soundtrack Version) Author Brady AuthorChannelId UCy95D_qGLG-eANxuO9DQEzA VideoId 6cWZWq5QQCQ Decision Different author channel TrackUrl https://music.youtube.com/watch?v=6cWZWq5QQCQ 7 { Rank = 8, Title = Future Isn't Sealed, Author = CAN'T STOP TALKING, AuthorChannelId = UCS2REA-szsJVdsSZJp2roHA, VideoId = H9YdBoeMgpg, Decision = KEEP, TrackUrl = https://music.youtube.com/watch?v=H9YdBoeMgpg } Rank 8 Title Future Isn't Sealed Author CAN'T STOP TALKING AuthorChannelId UCS2REA-szsJVdsSZJp2roHA VideoId H9YdBoeMgpg Decision KEEP TrackUrl https://music.youtube.com/watch?v=H9YdBoeMgpg 8 { Rank = 9, Title = Dem Can't Stop We from Talk (feat. Anthony B.), Author = Dubblestandart, AuthorChannelId = UCdOh54lOcTL-n7HYcng-B5Q, VideoId = YJ2CL4vWKpQ, Decision = Different author channel, TrackUrl = https://music.youtube.com/watch?v=YJ2CL4vWKpQ } Rank 9 Title Dem Can't Stop We from Talk (feat. Anthony B.) Author Dubblestandart Auth

App would propose 1 track link(s):


index value 0 { Platform = youtube_music, Url = https://music.youtube.com/watch?v=H9YdBoeMgpg, Label = Can't Stop Talking - Future Isn't Sealed } Platform youtube_music Url https://music.youtube.com/watch?v=H9YdBoeMgpg Label Can't Stop Talking - Future Isn't Sealed

## Full package response objects

Use these to inspect fields omitted from the tables, such as thumbnails and continuation data. Rerunning this cell makes no further requests. `null` means that search failed. These are the objects returned by YTMusicAPI after its parsing, not raw YouTube HTTP responses.


In [5]:
var jsonOptions = new JsonSerializerOptions { WriteIndented = true };
Console.WriteLine("ARTIST/CHANNEL RESPONSE");
Console.WriteLine(JsonSerializer.Serialize(artistResponse, jsonOptions));
Console.WriteLine("TRACK RESPONSE");
Console.WriteLine(JsonSerializer.Serialize(trackResponse, jsonOptions));


ARTIST/CHANNEL RESPONSE
{
  "Result": [
    {
      "Id": "UCS2REA-szsJVdsSZJp2roHA",
      "Url": "https://music.youtube.com/channel/UCS2REA-szsJVdsSZJp2roHA",
      "Title": "CAN\u0027T STOP TALKING",
      "Thumbnails": [
        {
          "Url": "https://yt3.googleusercontent.com/fc5e7S0FJbf3gtBHL6raDBaHpGxEb9zLKkR11Ny-hpdfSJ9VqxqKi64WIqfnJij9NJGnXcH1hzTL_ofK0Q=w60-h60-l90-rj",
          "Resolution": {
            "Width": 60,
            "Height": 60
          }
        },
        {
          "Url": "https://yt3.googleusercontent.com/fc5e7S0FJbf3gtBHL6raDBaHpGxEb9zLKkR11Ny-hpdfSJ9VqxqKi64WIqfnJij9NJGnXcH1hzTL_ofK0Q=w120-h120-l90-rj",
          "Resolution": {
            "Width": 120,
            "Height": 120
          }
        }
      ]
    },
    {
      "Id": "UCO4ec785eyIi_tD2GzrsyWQ",
      "Url": "https://music.youtube.com/channel/UCO4ec785eyIi_tD2GzrsyWQ",
      "Title": "CAN\u0027T STOP TALKING",
      "Thumbnails": [
        {
          "Url": "https://yt3.googleuser